In [ ]:
plan = """
iterate through (models x subsets x ks x norm_types) configurations
[x] aggregate results in different weight decomposition strategies
    1. whole layer
    2. mlp in layer
    3. specific weights in mlp: gate, up, down
    4. attn in layer: q, k, v, o
    
    then aggregate results across all strategies above
    -> save into ablation/{model}/aggregated-results/{subset}_k{k}_{norm_type}.tsv
[x] select the best strategies (k=3) from each of categories above and
    plot score distributions of decisive error steps vs. non- counterparts.
    -> save as one plot at ablation/{model}/{subset}_score-distribution.png
[x] plot dot chart (length, score). highlight score of gold step.
    -> save as one dot chart at ablation/{model}/{subset}_length-score-chart.png
[x] plot distance distribution (gold vs. predicted steps)
    -> save as one dot chart at ablation/{model}/{subset}_distance-chart.png   
[ ] compute agent prediction with aggregated score from trajectory.
[ ] our methods, remember, is currently a w/o gt setting. which beats agentracer.
[-] consider a reranking phase (LLM/get top-k, then sort temporally/...)
    [x] get top-k, then sort temporally: horrible results.
    [ ] LLM rerank
[x] longer contexts -> lower gradnorm? score spikes at VERY short context.
    investigate what kinds of those very short steps they are.
    [-] try to remove these steps from the rankings:
        > not good results
    ---------------------------------------------------------
    they are these kinds of steps:
    - FileSurfer: Address: file:///workspace/files
    Title: Download complete.
    Viewport position: Showing page 1 of 1.
    =======================
    # Download complete

    Saved file to '/workspace/files'
    - Orchestrator (thought): Next speaker WebSurfer
    - Orchestrator (thought): Next speaker WebSurfer
    - Orchestrator (thought): Next speaker WebSurfer
    - Orchestrator (termination condition): No agent selected.
    response = await client.create(messages)
    FINAL ANSWER: 0.00049
    SCENARIO.PY COMPLETE !#!#
    RUN.SH COMPLETE !#!#
    - Orchestrator (thought): Request satisfied.
    - Orchestrator (thought): Next speaker Assistant
    /usr/local/lib/python3.11/site-packages/autogen_magentic_one/agents/coder.py:55: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-05-13. Model mapping may be incorrect.
    response = await self._model_client.create(
    - Orchestrator (thought): Next speaker WebSurfer
    - Orchestrator (thought): Next speaker WebSurfer
    - Orchestrator (thought): Next speaker WebSurfer

[ ] clean up later 
    [ ] a test file for sal_extract 
    [ ] aggregate scripts in one folder.
    [ ] be careful with select_cotext. full context takes too long.

[ ] implement svd, rmb to compare with gradnorm config.
"""

In [63]:
from pathlib import Path
from ablation.core import (
    load_trajectories, get_param_names_and_sizes,
    discover_n_layers, build_strategies,
    CompiledConfigs, score_step,
)
from ablation.length_dist import context_word_count

# ── settings ──────────────────────────────────────────────────────────────────
MODEL      = "qwen3-8b"
SUBSET     = "hand-crafted"
STRATEGY   = "layer"            # which strategy to use
CONFIG     = "lm_head"          # which config within that strategy
NORM_TYPE  = "l1_norm"
MAX_CTX    = 500               # context length threshold (words)

# ── setup ─────────────────────────────────────────────────────────────────────
results_dir = Path("ablation") / MODEL / SUBSET
trajs       = load_trajectories(results_dir)
param_names, param_sizes = get_param_names_and_sizes(trajs)
strategies  = build_strategies(discover_n_layers(param_names))
cc          = CompiledConfigs.compile({CONFIG: strategies[STRATEGY][CONFIG]}, param_names, param_sizes)
cfg_idx     = 0  # single config → index 0

# ── collect ───────────────────────────────────────────────────────────────────
rows = []
for traj in trajs:
    filename     = traj["metadata"].get("filename", "")
    mistake_step = int(traj["metadata"]["mistake_step"])

    for log in traj["logs"]:
        if not log.get("statistics"):
            continue
        step_idx = int(log["step_idx"])
        role = traj["steps"][step_idx]['role']
        content = traj["steps"][step_idx]['content']
        ctx_len  = context_word_count(traj, step_idx)
        # if ctx_len < MAX_CTX:
        #     continue
        score = score_step(log, cc, NORM_TYPE)[cfg_idx]
        rows.append({
            "traj":       filename,
            "step_idx":   step_idx,
            "role":       role, 
            "content":    content, 
            "ctx_len":    ctx_len,
            "score":      float(score),
            "is_mistake": step_idx == mistake_step,
        })

print(f"Collected {len(rows)} steps with ctx_len ≤ {MAX_CTX}")

Collected 2935 steps with ctx_len ≤ 500


In [64]:
high_scores = sorted(rows, key=lambda x: x["score"], reverse=True)

In [57]:
lengths = [x['ctx_len'] for x in high_scores[:700]]
sum(lengths) / len(lengths)

346.04

In [58]:
len(high_scores[700:])

2235

In [59]:
len(high_scores)

2935

In [11]:
len([x for x in rows if x['is_mistake']])

47

In [12]:
scores = [x['score'] for x in rows]
sum(scores) / len(scores) * 1000000

2.422237374148488

In [76]:
import random
for i in range(10):
    x = random.choice(high_scores[:500])
    print(f"- {x['role']}: {x['content']}")

- FileSurfer: Address: file:///workspace/files
Title: Download complete.
Viewport position: Showing page 1 of 1.
# Download complete

Saved file to '/workspace/files'
- Orchestrator (thought): Next speaker WebSurfer
- Orchestrator (thought): Next speaker WebSurfer
- Orchestrator (thought): Next speaker WebSurfer
- Orchestrator (termination condition): No agent selected.
  response = await client.create(messages)
FINAL ANSWER: 0.00049
SCENARIO.PY COMPLETE !#!#
RUN.SH COMPLETE !#!#
- Orchestrator (thought): Request satisfied.
- Orchestrator (thought): Next speaker Assistant
/usr/local/lib/python3.11/site-packages/autogen_magentic_one/agents/coder.py:55: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-05-13. Model mapping may be incorrect.
  response = await self._model_client.create(
- Orchestrator (thought): Next speaker WebSurfer
- Orchestrator (thought): Next speaker WebSurfer
- Orchestrator (thought): Next speaker WebSurfer


In [85]:
import pandas as pd
top = pd.read_csv("ablation/llama-3.1-8b/aggregated-results/hand-crafted_k1_l2_norm.tsv", sep="\t")
top_str = top.sort_values(
    ["step_acc", "agent_acc", "config"], 
    ascending=[False, False, False],
).to_string()
print(top_str)

         strategy        config  step_acc  agent_acc
0    attn_weights          k/17    0.2414     0.6034
1    attn_weights          q/22    0.2414     0.5862
33    mlp_weights       down/23    0.2241     0.6724
32    mlp_weights         up/17    0.2241     0.6379
21   attn_weights          o/19    0.2241     0.6379
23   attn_weights          o/18    0.2241     0.6379
2           layer       lm_head    0.2241     0.6379
16    mlp_weights       gate/17    0.2241     0.6379
30           attn       attn/18    0.2241     0.6379
34    mlp_weights         up/14    0.2241     0.6207
4             mlp        mlp/14    0.2241     0.6207
26    mlp_weights        down/6    0.2241     0.6207
24    mlp_weights       down/14    0.2241     0.6207
25    mlp_weights       down/10    0.2241     0.6207
15   attn_weights          q/20    0.2241     0.6034
14   attn_weights          o/22    0.2241     0.6034
9    attn_weights          o/16    0.2241     0.6034
10   attn_weights          o/15    0.2241     

## task 1

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ── strategy definitions ──────────────────────────────────────────────────────
def build_strategies(L: int) -> dict[str, dict[str, str]]:
    def per_layer(prefix, suffix=""):
        return {f"{prefix}/{i}": rf"model\.layers\.{i}\.{suffix}" for i in range(L)}

    return {
        "layer": {
            **per_layer("layer"),
            "lm_head":     r"lm_head\.",
            "embed_tokens": r"model\.embed_tokens\.",
        },
        "mlp":         per_layer("mlp",  r"mlp\."),
        "mlp_weights": {
            **per_layer("gate", r"mlp\.gate_proj\."),
            **per_layer("up",   r"mlp\.up_proj\."),
            **per_layer("down", r"mlp\.down_proj\."),
        },
        "attn":        per_layer("attn", r"self_attn\."),
        "attn_weights": {
            **per_layer("q", r"self_attn\.q_proj\."),
            **per_layer("k", r"self_attn\.k_proj\."),
            **per_layer("v", r"self_attn\.v_proj\."),
            **per_layer("o", r"self_attn\.o_proj\."),
        },
    }


# ── data loading ──────────────────────────────────────────────────────────────
def load_trajectories(results_dir: Path) -> list[dict]:
    return [json.loads(f.read_text()) for f in sorted(results_dir.glob("*.json"))]


def get_param_names_and_sizes(trajectories: list[dict]) -> tuple[list[str], np.ndarray]:
    sample_stats = next(
        log["statistics"]
        for data in trajectories
        for log in data["logs"]
        if log.get("statistics")
    )
    param_names = list(sample_stats.keys())
    param_sizes = np.array([sample_stats[p]["n_params"] for p in param_names], dtype=np.float64)
    return param_names, param_sizes


def discover_n_layers(param_names: list[str]) -> int:
    """Infer number of layers from the highest layer index in param names."""
    indices = [
        int(m.group(1))
        for p in param_names
        if (m := re.search(r"model\.layers\.(\d+)\.", p))
    ]
    if not indices:
        raise ValueError("Could not discover n_layers: no 'model.layers.N.' params found.")
    return max(indices) + 1


# ── scoring ───────────────────────────────────────────────────────────────────
def score_log(
    log: dict,
    param_names: list[str],
    mask_matrix: np.ndarray,
    n_params_per_config: np.ndarray,
    norm_type: str,
) -> np.ndarray:
    """
    Compute a normalised gradient norm score for each parameter group in one log entry.

    For each group, aggregates per-parameter norms across all parameters in the group,
    then normalises by the group's total parameter count:
      - l1_norm: mean absolute gradient  = sum(|w|)  / N
      - l2_norm: RMS gradient norm       = sqrt(sum(w²)) / N

    Args:
        log:                 A single log entry containing a "statistics" dict keyed by
                             parameter name, each with "l1_norm" or "l2_norm_sq" fields.
        param_names:         Ordered list of parameter names matching the stats keys.
        mask_matrix:         Boolean matrix of shape (n_configs, n_params) where entry
                             [i, j] is True if parameter j belongs to group i.
        n_params_per_config: Total parameter count per group, shape (n_configs,).
        norm_type:           One of "l1_norm" or "l2_norm".

    Returns:
        Normalised scores of shape (n_configs,). Groups with no parameters are NaN.
    """
    stats = log["statistics"]
    safe_counts = np.where(n_params_per_config > 0, n_params_per_config, np.nan)

    with np.errstate(invalid="ignore"):
        if norm_type == "l2_norm":
            vals = np.array([stats[p]["l2_norm_sq"] for p in param_names], dtype=np.float64)
            return np.sqrt((mask_matrix @ vals)) / safe_counts
        else:
            vals = np.array([stats[p]["l1_norm"] for p in param_names], dtype=np.float64)
            return (mask_matrix @ vals) / safe_counts


# ── per-trajectory evaluation ─────────────────────────────────────────────────
def evaluate_trajectory(
    data: dict,
    param_names: list[str],
    mask_matrix: np.ndarray,
    n_params_per_config: np.ndarray,
    norm_type: str,
    k: int,
) -> tuple[np.ndarray, np.ndarray] | None:
    """Returns (step_correct, agent_correct) of shape (n_configs,), or None if no valid logs."""
    meta          = data["metadata"]
    mistake_step  = int(meta["mistake_step"])
    mistake_agent = meta["mistake_agent"]
    step_roles    = {s["step_idx"]: s["role"] for s in data["steps"]}

    valid_logs = [log for log in data["logs"] if log.get("statistics")]
    if not valid_logs:
        return None

    score_matrix     = np.stack([
        score_log(log, param_names, mask_matrix, n_params_per_config, norm_type)
        for log in valid_logs
    ])                                                        # (n_logs, n_configs)
    step_indices     = np.array([log["step_idx"] for log in valid_logs])
    pred_step_matrix = step_indices[np.argsort(score_matrix, axis=0)[:k]]  # (k, n_configs)

    step_correct  = np.any(pred_step_matrix == mistake_step, axis=0).astype(float)
    agent_correct = np.array([
        float(mistake_agent in [
            "Orchestrator" if "orchestrator" in step_roles.get(idx, "").lower() else step_roles.get(idx, "unknown")
            for idx in pred_step_matrix[:, c]
        ])
        for c in range(mask_matrix.shape[0])
    ])
    return step_correct, agent_correct


# ── strategy sweep ────────────────────────────────────────────────────────────
def run_strategy(
    trajectories: list[dict],
    config_dict: dict[str, str],
    param_names: list[str],
    param_sizes: np.ndarray,
    norm_type: str,
    k: int,
) -> pd.DataFrame:
    """Sweep all trajectories for one strategy. Returns DataFrame: config, step_acc, agent_acc."""
    config_names    = list(config_dict.keys())
    config_patterns = list(config_dict.values())
    n_configs       = len(config_names)

    mask_matrix = np.array(
        [[bool(re.search(pat, p)) for p in param_names] for pat in config_patterns],
        dtype=np.float64,
    )
    n_params_per_config = mask_matrix @ param_sizes

    step_correct_sum  = np.zeros(n_configs)
    agent_correct_sum = np.zeros(n_configs)
    n_total           = 0

    for data in trajectories:
        result = evaluate_trajectory(
            data, param_names, mask_matrix, n_params_per_config, norm_type, k
        )
        if result is None:
            continue
        step_correct, agent_correct = result
        step_correct_sum  += step_correct
        agent_correct_sum += agent_correct
        n_total           += 1

    denom = n_total if n_total > 0 else 1
    return pd.DataFrame({
        "config":    config_names,
        "step_acc":  step_correct_sum  / denom,
        "agent_acc": agent_correct_sum / denom,
    })


# ── main ──────────────────────────────────────────────────────────────────────
def sweep(
    model:       str,
    subset:      str,
    norm_type:   str,
    k:           int,
    verbose: bool = False,
):
    results_dir  = Path("ablation") / model / subset
    trajectories = load_trajectories(results_dir)
    param_names, param_sizes = get_param_names_and_sizes(trajectories)
    n_layers     = discover_n_layers(param_names)
    if verbose: print(f"Discovered {n_layers} layers.")
    strategies   = build_strategies(n_layers)

    out_dir = Path(f"ablation/{model}/aggregated-results")
    out_dir.mkdir(parents=True, exist_ok=True)
    agg_path = out_dir / f"{subset}_k{k}_{norm_type}.tsv"

    strategy_dfs = {}
    for name, config_dict in strategies.items():
        if verbose: print(f"Running strategy: {name} ({len(config_dict)} configs)...")
        df = run_strategy(trajectories, config_dict, param_names, param_sizes, norm_type, k)
        strategy_dfs[name] = df

    agg = (
        pd.concat([df.assign(strategy=name) for name, df in strategy_dfs.items()], ignore_index=True)
        .sort_values("step_acc", ascending=False)
        .reset_index(drop=True)
        [["strategy", "config", "step_acc", "agent_acc"]]
    )

    agg.to_csv(agg_path, sep="\t", index=False, float_format="%.4f")
    return agg

In [59]:
NORM_TYPES   = ["l1_norm", "l2_norm"]
KS          = [1, 3, 5, 10]
MODELS       = ["llama-3.1-8b", "qwen3-8b"]
SUBSETS      = ["algorithm-generated", "hand-crafted"]

from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def sweep_unpacked(args):
    norm_type, k, model, subset = args
    return args, sweep(model=model, subset=subset, norm_type=norm_type, k=k)

combos = list(product(NORM_TYPES, KS, MODELS, SUBSETS))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(sweep_unpacked, combo): combo for combo in combos}
    results = {}
    for future in tqdm(as_completed(futures), total=len(combos)):
        args, agg = future.result()
        results[args] = agg

100%|██████████| 32/32 [00:12<00:00,  2.64it/s]


In [58]:
from itertools import product

NORM_TYPES   = ["l1_norm", "l2_norm"]
KS          = [1, 3, 5, 10]
MODELS       = ["llama-3.1-8b", "qwen3-8b"]
SUBSETS      = ["algorithm-generated", "hand-crafted"]

for norm_type, k, model, subset in product(NORM_TYPES, KS, MODELS, SUBSETS):
    agg = sweep(model=model, subset=subset, norm_type=norm_type, k=k)

## task 2

In [72]:
import pandas as pd

file_path = "ablation/qwen3-8b/aggregated-results/hand-crafted_k3_l1_norm.tsv"
df = pd.read_csv(file_path, sep="\t")


In [ ]:
"""[ ] select the best strategies (k=3) from each of categories above and
    plot score distributions of decisive error steps vs. non- counterparts.
    -> save as one plot at ablation/{model}/{subset}_score-distribution.png"""

In [73]:
x = df[df["strategy"] == "attn_weights"].sort_values(
    ["step_acc", "config"],
    ascending=[False, False]
)
print(x.to_string())

         strategy config  step_acc  agent_acc
0    attn_weights   q/30    0.3621     0.8103
1    attn_weights   k/30    0.3621     0.7931
3    attn_weights   v/25    0.3448     0.7931
2    attn_weights   o/35    0.3448     0.8103
6    attn_weights   o/25    0.3448     0.7931
7    attn_weights   o/24    0.3448     0.7931
10   attn_weights   v/30    0.3276     0.8103
11   attn_weights   v/28    0.3276     0.7931
36   attn_weights   q/28    0.3276     0.7586
25   attn_weights   q/25    0.3276     0.7931
40   attn_weights   o/27    0.3276     0.7759
42   attn_weights   o/23    0.3276     0.7586
28   attn_weights   k/28    0.3276     0.7759
29   attn_weights   k/25    0.3276     0.7759
54   attn_weights   v/35    0.3103     0.8276
56   attn_weights   v/24    0.3103     0.7759
59   attn_weights   v/23    0.3103     0.7414
53   attn_weights   q/27    0.3103     0.7759
73   attn_weights   q/24    0.3103     0.7586
74   attn_weights   q/21    0.3103     0.7414
46   attn_weights   q/20    0.3103

In [1]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from scipy.stats import gaussian_kde

# ── config ────────────────────────────────────────────────────────────────────
MODEL      = "qwen3-8b"          # change to your model name
SUBSET     = "hand-crafted"      # change to your subset name
K_SWEEP    = 1                   # the k used when running sweep()  (for TSV filename)
K_TOPCONFIGS = 3                 # how many top configs to display per strategy
KDE_POINTS = 400
STRATEGIES = ["layer", "mlp", "attn", "mlp_weights", "attn_weights"]
NORM_TYPES = ["l1_norm", "l2_norm"]

RESULTS_DIR = Path("ablation") / MODEL / SUBSET
AGG_DIR     = Path("ablation") / MODEL / "aggregated-results"

# ── helpers from sweep.py (inlined to keep this script self-contained) ────────
def build_strategies(L: int) -> dict[str, dict[str, str]]:
    def per_layer(prefix, suffix=""):
        return {f"{prefix}/{i}": rf"model\.layers\.{i}\.{suffix}" for i in range(L)}
    return {
        "layer": {
            **per_layer("layer"),
            "lm_head":      r"lm_head\.",
            "embed_tokens": r"model\.embed_tokens\.",
        },
        "mlp":         per_layer("mlp",  r"mlp\."),
        "mlp_weights": {
            **per_layer("gate", r"mlp\.gate_proj\."),
            **per_layer("up",   r"mlp\.up_proj\."),
            **per_layer("down", r"mlp\.down_proj\."),
        },
        "attn":        per_layer("attn", r"self_attn\."),
        "attn_weights": {
            **per_layer("q", r"self_attn\.q_proj\."),
            **per_layer("k", r"self_attn\.k_proj\."),
            **per_layer("v", r"self_attn\.v_proj\."),
            **per_layer("o", r"self_attn\.o_proj\."),
        },
    }


def load_trajectories(results_dir: Path) -> list[dict]:
    return [json.loads(f.read_text()) for f in sorted(results_dir.glob("*.json"))]


def get_param_names_and_sizes(trajectories: list[dict]) -> tuple[list[str], np.ndarray]:
    sample_stats = next(
        log["statistics"]
        for data in trajectories
        for log in data["logs"]
        if log.get("statistics")
    )
    param_names = list(sample_stats.keys())
    param_sizes = np.array([sample_stats[p]["n_params"] for p in param_names], dtype=np.float64)
    return param_names, param_sizes


def discover_n_layers(param_names: list[str]) -> int:
    indices = [
        int(m.group(1))
        for p in param_names
        if (m := re.search(r"model\.layers\.(\d+)\.", p))
    ]
    if not indices:
        raise ValueError("Could not discover n_layers.")
    return max(indices) + 1


def score_log_for_configs(
    log: dict,
    param_names: list[str],
    mask_matrix: np.ndarray,          # (n_configs, n_params)
    n_params_per_config: np.ndarray,   # (n_configs,)
    norm_type: str,
) -> np.ndarray:                       # (n_configs,)
    """Score one log entry for each config in mask_matrix."""
    stats = log["statistics"]
    safe_counts = np.where(n_params_per_config > 0, n_params_per_config, np.nan)
    with np.errstate(invalid="ignore"):
        if norm_type == "l2_norm":
            vals = np.array([stats[p]["l2_norm_sq"] for p in param_names], dtype=np.float64)
            return np.sqrt(mask_matrix @ vals) / safe_counts
        else:
            vals = np.array([stats[p]["l1_norm"] for p in param_names], dtype=np.float64)
            return (mask_matrix @ vals) / safe_counts


# ── step 1: load aggregated TSV and pick top-K configs per strategy ───────────
def load_top_configs(norm_type: str, k_top: int) -> dict[str, list[str]]:
    """
    Returns {strategy_name: [config_name, ...]} with k_top best configs
    ranked by step_acc from the pre-computed TSV.
    """
    tsv_path = AGG_DIR / f"{SUBSET}_k{K_SWEEP}_{norm_type}.tsv"
    df = pd.read_csv(tsv_path, sep="\t")

    top_configs: dict[str, list[str]] = {}
    for strat in STRATEGIES:
        sub = (
            df[df["strategy"] == strat]
            .sort_values("step_acc", ascending=False)
            .head(k_top)
        )
        top_configs[strat] = sub["config"].tolist()
    return top_configs


# ── step 2: collect per-step scores for selected configs ─────────────────────
def collect_scores(
    trajectories: list[dict],
    param_names: list[str],
    param_sizes: np.ndarray,
    all_strategies: dict[str, dict[str, str]],
    top_configs_by_norm: dict[str, dict[str, list[str]]],  # norm → strat → [configs]
) -> dict[str, dict[str, dict[str, dict[str, list[float]]]]]:
    """
    Returns nested dict:
      norm_type → strategy → config → {"mistake": [...], "normal": [...]}
    Only the selected top-K configs per (norm, strategy) are scored.
    """
    # Pre-build mask matrices for every (norm, strategy, config) we need
    # Structure: needed_configs[norm][strat] = list[str]
    # We batch all configs in a strategy into one mask_matrix call per log.

    # Initialise score storage
    score_store: dict = {
        nt: {
            strat: {
                cfg: {"mistake": [], "normal": []}
                for cfg in top_configs_by_norm[nt][strat]
            }
            for strat in STRATEGIES
        }
        for nt in NORM_TYPES
    }

    # Pre-compute mask matrices: (norm, strat) → (mask_matrix, config_names, n_params_per_config)
    mask_cache: dict = {}
    for nt in NORM_TYPES:
        for strat in STRATEGIES:
            selected_cfgs = top_configs_by_norm[nt][strat]
            strat_dict    = all_strategies[strat]
            patterns      = [strat_dict[c] for c in selected_cfgs]
            mask_matrix   = np.array(
                [[bool(re.search(pat, p)) for p in param_names] for pat in patterns],
                dtype=np.float64,
            )
            n_params_per_config = mask_matrix @ param_sizes
            mask_cache[(nt, strat)] = (mask_matrix, selected_cfgs, n_params_per_config)

    # Sweep trajectories once; score all needed (norm, strat) combos per log
    for data in trajectories:
        mistake_step = int(data["metadata"]["mistake_step"])

        for log in data["logs"]:
            if not log.get("statistics"):
                continue
            step = int(log["step_idx"])
            kind = "mistake" if step == mistake_step else "normal"

            for nt in NORM_TYPES:
                for strat in STRATEGIES:
                    mask_matrix, cfg_names, n_params = mask_cache[(nt, strat)]
                    scores = score_log_for_configs(
                        log, param_names, mask_matrix, n_params, nt
                    )
                    for i, cfg in enumerate(cfg_names):
                        val = scores[i]
                        if not np.isnan(val):
                            score_store[nt][strat][cfg][kind].append(float(val))

    return score_store


# ── step 3: KDE plot helper ───────────────────────────────────────────────────
def plot_kde(ax: plt.Axes, values: list[float], label: str, color: str) -> None:
    if len(values) < 2:
        return
    arr = np.array(values)
    kde = gaussian_kde(arr)
    xs  = np.linspace(arr.min(), arr.max(), KDE_POINTS)
    ax.plot(xs, kde(xs), label=label, color=color, linewidth=1.2)
    ax.fill_between(xs, kde(xs), alpha=0.18, color=color)


# ── step 4: build the figure ──────────────────────────────────────────────────
def make_figure(
    score_store: dict,
    top_configs_by_norm: dict,
) -> plt.Figure:
    n_strat = len(STRATEGIES)
    n_cols  = K_TOPCONFIGS

    # Two halves: top = L1-norm ranking, bottom = L2-norm ranking
    # Each half has n_strat rows × n_cols cols
    total_rows = 2 * n_strat
    fig, axes  = plt.subplots(
        nrows   = total_rows,
        ncols   = n_cols,
        figsize = (n_cols * 3.4, total_rows * 2.4),
        squeeze = False,
    )

    half_labels = {
        "l1_norm": "Ranked by L1-norm",
        "l2_norm": "Ranked by L2-norm",
    }
    half_offsets = {"l1_norm": 0, "l2_norm": n_strat}   # row offset per norm block

    for nt in NORM_TYPES:
        row_offset = half_offsets[nt]

        for si, strat in enumerate(STRATEGIES):
            row       = row_offset + si
            cfg_list  = top_configs_by_norm[nt][strat]   # top-K configs for this (norm, strat)

            for ci, cfg in enumerate(cfg_list):
                ax = axes[row, ci]
                plot_kde(ax, score_store[nt][strat][cfg]["normal"],  "normal",  "steelblue")
                plot_kde(ax, score_store[nt][strat][cfg]["mistake"], "mistake", "tomato")

                # Rank label (1-indexed) + config name
                ax.set_title(f"#{ci+1} {cfg}", fontsize=7)
                ax.tick_params(labelsize=6)
                ax.set_xlabel("score", fontsize=6)
                if ci == 0:
                    # Left-most column: show strategy name as y-label
                    ax.set_ylabel(f"[{strat}]\ndensity", fontsize=6)

            # Hide unused columns if fewer than K_TOPCONFIGS configs returned
            for ci in range(len(cfg_list), n_cols):
                axes[row, ci].set_visible(False)

        # Section title spanning the full width for this norm block
        # Use a centered suptitle-style text via a hidden spine row title
        mid_row = row_offset + n_strat // 2
        axes[mid_row, 0].annotate(
            half_labels[nt],
            xy=(-0.35, 0.5), xycoords="axes fraction",
            fontsize=8, fontweight="bold", rotation=90,
            va="center", ha="center", color="dimgray",
        )

    # Shared legend (top-right)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", fontsize=9, framealpha=0.8)

    # Half-divider line between the two norm blocks
    # Draw a horizontal line between row n_strat-1 and n_strat
    line_y = 1.0 - n_strat / total_rows   # normalised figure coordinate
    fig.add_artist(
        plt.Line2D([0.01, 0.99], [line_y, line_y],
                   transform=fig.transFigure, color="gray",
                   linewidth=1.2, linestyle="--")
    )

    fig.suptitle(
        f"Grad-norm KDE: normal vs mistake  |  model={MODEL}  subset={SUBSET}\n"
        f"Top-{K_TOPCONFIGS} configs per strategy (ranked by step accuracy)",
        fontsize=10, y=1.01,
    )
    plt.tight_layout()
    return fig


# ── main ──────────────────────────────────────────────────────────────────────
def main():
    print("Loading trajectories …")
    trajectories             = load_trajectories(RESULTS_DIR)
    param_names, param_sizes = get_param_names_and_sizes(trajectories)
    n_layers                 = discover_n_layers(param_names)
    print(f"  {len(trajectories)} trajectories, {n_layers} layers discovered.")

    all_strategies = build_strategies(n_layers)

    # Load top-K configs for each norm type from the pre-computed TSV files
    print("Loading top configs from aggregated TSVs …")
    top_configs_by_norm = {nt: load_top_configs(nt, K_TOPCONFIGS) for nt in NORM_TYPES}

    for nt in NORM_TYPES:
        for strat in STRATEGIES:
            cfgs = top_configs_by_norm[nt][strat]
            print(f"  [{nt}] {strat}: {cfgs}")

    # Collect per-step scores for all selected configs in one trajectory pass
    # print("Collecting scores …")
    score_store = collect_scores(
        trajectories, param_names, param_sizes,
        all_strategies, top_configs_by_norm,
    )

    # Plot
    # print("Plotting …")
    fig = make_figure(score_store, top_configs_by_norm)

    out_path = Path("ablation") / MODEL / "aggregated-results" / \
               f"{SUBSET}_score-distribution.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved → {out_path}")
    # plt.show()

# if __name__ == "__main__":
#     main()

## svd

In [1]:
from __future__ import annotations
 
import argparse
import json
from dataclasses import dataclass, field
from pathlib import Path
 
import torch
import numpy as np
from tqdm import tqdm
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Data structures
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class StepIndex:
    """Row-level metadata for one entry in the stacked gradient matrix."""
    row:        int     # row index in G
    traj_idx:   int     # index into the loaded data list
    step_idx:   int     # step index within the trajectory
    is_mistake: bool    # whether this is the gold mistake step
 
 
@dataclass
class GradientStore:
    """All gradient data stacked into a single matrix with index mappings.
 
    Attributes
    ----------
    G           : (T, d) float32 tensor — all raw gradient vectors.
    index       : list[StepIndex] of length T — per-row metadata.
    traj_meta   : list[dict] — per-trajectory metadata dicts.
    traj_ranges : list[tuple[int,int]] — (start_row, end_row) for each traj.
    device      : torch device where G lives.
    """
    G:           torch.Tensor
    index:       list[StepIndex]
    traj_meta:   list[dict]
    traj_ranges: list[tuple[int, int]]
    device:      torch.device
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Loading: files → single stacked matrix on GPU
# ─────────────────────────────────────────────────────────────────────────────
 
def load_and_stack(
    input_dir: Path, 
    device: torch.device,
    proj_matrix: torch.Tensor | None = None
) -> GradientStore:
    """Load all .pt gradient files, optionally project them, and stack onto device."""
    files = sorted(input_dir.glob("*.pt"))
    files = [f for f in files if f.name != "config.pt"]
    if not files:
        raise FileNotFoundError(f"No .pt gradient files in {input_dir}")

    all_grads: list[torch.Tensor] = []
    index: list[StepIndex] = []
    traj_meta: list[dict] = []
    traj_ranges: list[tuple[int, int]] = []

    row = 0
    for traj_idx, fp in enumerate(tqdm(files, desc="Loading .pt files")):
        payload = torch.load(fp, map_location="cpu", weights_only=False)
        metadata = payload["metadata"]
        gradients = payload["gradients"]
        mistake_step = int(metadata["mistake_step"])

        traj_meta.append(metadata)
        start_row = row

        for step_idx in sorted(int(k) for k in gradients.keys()):
            grad_vec = gradients[step_idx]   # original shape: (d,)
            
            # --- APPLY RANDOM PROJECTION HERE ---
            if proj_matrix is not None:
                # Cast grad_vec to match proj_matrix dtype (usually float32)
                grad_vec = grad_vec.to(proj_matrix.dtype)
                # Matrix multiplication: (d,) @ (d, k) -> (k,)
                grad_vec = torch.matmul(grad_vec, proj_matrix)
                
            all_grads.append(grad_vec)
            index.append(StepIndex(
                row=row,
                traj_idx=traj_idx,
                step_idx=step_idx,
                is_mistake=(step_idx == mistake_step),
            ))
            row += 1

        traj_ranges.append((start_row, row))

    T = len(all_grads)
    d_final = all_grads[0].shape[0]
    print(f"Stacking {T} vectors × d={d_final:,} → ({T}, {d_final})")

    G = torch.stack(all_grads, dim=0).to(dtype=torch.float32, device=device)
    mem_gb = G.element_size() * G.numel() / 1e9
    print(f"  G on {device}: {mem_gb:.2f} GB")

    return GradientStore(
        G=G, index=index, traj_meta=traj_meta,
        traj_ranges=traj_ranges, device=device,
    )

In [2]:
INPUT_DIR = Path("sal_outputs/grads/qwen3-8b/hand-crafted")
DEVICE = torch.device("cpu")

store = load_and_stack(INPUT_DIR, DEVICE)
print(f"Loaded {len(store.traj_meta)} trajectories, "
        f"{store.G.shape[0]} total steps.")

Loading .pt files:   0%|          | 0/58 [00:00<?, ?it/s]

Loading .pt files: 100%|██████████| 58/58 [00:27<00:00,  2.12it/s]


Stacking 2935 vectors × d=4,194,304 → (2935, 4194304)
  G on cpu: 49.24 GB
Loaded 58 trajectories, 2935 total steps.


In [3]:
store.index[1]

StepIndex(row=1, traj_idx=0, step_idx=2, is_mistake=False)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Reference gradient (vectorised on GPU)
# ─────────────────────────────────────────────────────────────────────────────
 
def compute_reference_gradient(
    store:    GradientStore,
    strategy: str = "all",
) -> torch.Tensor:
    """Compute ∇̄ on GPU using boolean masking for the chosen strategy.
 
    Returns (d,) float32 tensor on the same device as store.G.
    """
    if strategy == "all":
        ref_grad = store.G.mean(dim=0)
        n = store.G.shape[0]
    elif strategy == "pre_mistake":
        mask = torch.tensor(
            [entry.step_idx < int(
                store.traj_meta[entry.traj_idx]["mistake_step"]
            ) for entry in store.index],
            dtype=torch.bool, device=store.device,
        )
        n = mask.sum().item()
        if n == 0:
            raise RuntimeError("No pre-mistake steps found.")
        ref_grad = store.G[mask].mean(dim=0)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
 
    print(f"  Reference gradient: strategy={strategy}, "
          f"n_steps={n}, ‖∇̄‖₁={ref_grad.abs().sum().item():.4f}")
    return ref_grad

In [5]:
ref_grad = compute_reference_gradient(store, strategy="pre_mistake")

  Reference gradient: strategy=pre_mistake, n_steps=821, ‖∇̄‖₁=973.9730


In [6]:
ref_grad.min(), ref_grad.max()

(tensor(-0.5637), tensor(0.6204))

In [7]:
def center_and_svd(
    store:        GradientStore,
    ref_grad:     torch.Tensor,
    n_components: int = 1,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Subtract ref_grad from G in-place, then compute top singular vector(s).
 
    After this call, store.G contains centered vectors.
 
    Returns
    -------
    (V, S)
        V : (d,) or (d, n_components) — top singular vector(s).
        S : (n_components,)           — corresponding singular values.
    """
    # ── Center in-place ──────────────────────────────────────────────
    store.G -= ref_grad.unsqueeze(0)  # broadcast (T, d) - (1, d)
    print(f"  Centered G in-place.")
 
    # ── Randomised SVD on GPU ────────────────────────────────────────
    print(f"  Running torch.svd_lowrank(q={n_components}, niter=5) "
          f"on {store.device} ...")
    U, S, V = torch.svd_lowrank(store.G, q=n_components, niter=5)
    # V: (d, n_components),  S: (n_components,)
 
    print(f"  Top {n_components} singular value(s): "
          f"{[f'{s:.4f}' for s in S.tolist()]}")
 
    if n_components == 1:
        return V.squeeze(1), S
    return V, S

In [16]:
v, singular_values = center_and_svd(
    store, ref_grad, n_components=5,
)

  Centered G in-place.
  Running torch.svd_lowrank(q=5, niter=5) on cpu ...
  Top 5 singular value(s): ['196.6483', '37.4557', '33.8286', '25.5507', '17.2110']


In [9]:
v[:, 0]

tensor([ 5.8375e-05, -2.6031e-05,  1.1888e-05,  ..., -2.1641e-05,
        -4.5781e-05,  1.6112e-04])

In [17]:
def compute_sal_scores(
    store: GradientStore,
    v:     torch.Tensor,
) -> torch.Tensor:
    """Compute τ = (G @ v)² for all steps in one operation.
 
    Assumes store.G is already centered.
 
    Returns (T,) tensor of SAL scores on the same device.
    """
    projections = store.G @ v        # (T,)
    scores = projections.square()    # (T,)
    return scores


In [18]:
scores = compute_sal_scores(store, v[:, 0])

In [12]:
def format_results(
    store:  GradientStore,
    scores: torch.Tensor,
) -> list[dict]:
    """Convert SAL scores into per-trajectory JSON dicts.
 
    Output format matches evaluate_gradnorm.py expectations:
    each trajectory gets a dict with "metadata", "steps", "logs".
    The SAL score appears under l1_norm/l2_norm with key "sal".
    """
    scores_cpu = scores.cpu().tolist()
    n_trajs = len(store.traj_meta)
    results = []
 
    for traj_idx in range(n_trajs):
        start, end = store.traj_ranges[traj_idx]
        metadata = store.traj_meta[traj_idx]
 
        logs = []
        for row in range(start, end):
            entry = store.index[row]
            sc = scores_cpu[row]
            logs.append({
                "step_idx":  entry.step_idx,
                "sal_score": sc,
                # "l1_norm":   {"sal": sc},
                # "l2_norm":   {"sal": sc},
            })
 
        results.append({
            "metadata": metadata,
            "steps":    [],
            "logs":     logs,
        })
 
    return results

In [ ]:
def step_at_k(results, k, reverse=False):
    correct = 0
    count   = 0
    for example in results:
        mistake_step = int(example['metadata']['mistake_step'])
        ranked = sorted(example['logs'], key = lambda x: x['sal_score'], reverse=reverse)[:k]
        ranked_idxs = [x['step_idx'] for x in ranked]
        correct += int(mistake_step in ranked_idxs)
        count += 1
    return correct / count



In [19]:
results = format_results(store, scores)

In [20]:
for k in [1, 3, 5, 10]:
    print(f"step@{k} descending:", step_at_k(results[:], k, reverse=True))
    print(f"step@{k} ascending: ", step_at_k(results[:], k, reverse=False))
    print("--" * 20)

step@1 descending: 0.20689655172413793
step@1 ascending:  0.017241379310344827
----------------------------------------
step@3 descending: 0.29310344827586204
step@3 ascending:  0.05172413793103448
----------------------------------------
step@5 descending: 0.39655172413793105
step@5 ascending:  0.1206896551724138
----------------------------------------
step@10 descending: 0.5172413793103449
step@10 ascending:  0.13793103448275862
----------------------------------------


In [15]:
for k in [1, 3, 5, 10]:
    print(f"step@{k} descending:", step_at_k(results[:], k, reverse=True))
    print(f"step@{k} ascending: ", step_at_k(results[:], k, reverse=False))
    print("--" * 20)

step@1 descending: 0.017241379310344827
step@1 ascending:  0.0
----------------------------------------
step@3 descending: 0.10344827586206896
step@3 ascending:  0.06896551724137931
----------------------------------------
step@5 descending: 0.20689655172413793
step@5 ascending:  0.10344827586206896
----------------------------------------
step@10 descending: 0.3448275862068966
step@10 ascending:  0.27586206896551724
----------------------------------------


In [29]:
for k in [1, 3, 5, 10]:
    print(f"step@{k} descending:", step_at_k(results[:], k, reverse=True))
    print(f"step@{k} ascending: ", step_at_k(results[:], k, reverse=False))
    print("--" * 20)

step@1 descending: 0.20689655172413793
step@1 ascending:  0.017241379310344827
----------------------------------------
step@3 descending: 0.29310344827586204
step@3 ascending:  0.05172413793103448
----------------------------------------
step@5 descending: 0.39655172413793105
step@5 ascending:  0.1206896551724138
----------------------------------------
step@10 descending: 0.5172413793103449
step@10 ascending:  0.13793103448275862
----------------------------------------


In [15]:
for k in [1, 3, 5, 10]:
    print(f"step@{k} descending:", step_at_k(results[:], k, reverse=True))
    print(f"step@{k} ascending: ", step_at_k(results[:], k, reverse=False))
    print("--" * 20)

step@1 descending: 0.017241379310344827
step@1 ascending:  0.0
----------------------------------------
step@3 descending: 0.10344827586206896
step@3 ascending:  0.06896551724137931
----------------------------------------
step@5 descending: 0.20689655172413793
step@5 ascending:  0.10344827586206896
----------------------------------------
step@10 descending: 0.3448275862068966
step@10 ascending:  0.27586206896551724
----------------------------------------


In [109]:
for k in [1, 3, 5, 10]:
    print(f"step@{k} descending:", step_at_k(results[:], k, reverse=True))
    print(f"step@{k} ascending: ", step_at_k(results[:], k, reverse=False))
    print("--" * 20)

step@1 descending: 0.017241379310344827
step@1 ascending:  0.0
----------------------------------------
step@3 descending: 0.10344827586206896
step@3 ascending:  0.06896551724137931
----------------------------------------
step@5 descending: 0.22413793103448276
step@5 ascending:  0.1206896551724138
----------------------------------------
step@10 descending: 0.3448275862068966
step@10 ascending:  0.2413793103448276
----------------------------------------


In [64]:
example = results[8]
mistake_step = example['metadata']['mistake_step']
print('mistake step :', int(mistake_step))
mistake_score = [x for x in example['logs'] if x['step_idx'] == int(mistake_step)][0]
print('mistake score:', mistake_score)
ranked = sorted(example['logs'], key = lambda x: x['sal_score'])
ranked[:10]

mistake step : 8
mistake score: {'step_idx': 8, 'sal_score': 5.319550514221191}


[{'step_idx': 3, 'sal_score': 0.06741652637720108},
 {'step_idx': 26, 'sal_score': 0.41431671380996704},
 {'step_idx': 34, 'sal_score': 0.6010156273841858},
 {'step_idx': 30, 'sal_score': 0.7347960472106934},
 {'step_idx': 22, 'sal_score': 1.279832124710083},
 {'step_idx': 10, 'sal_score': 1.337388277053833},
 {'step_idx': 18, 'sal_score': 1.484771966934204},
 {'step_idx': 6, 'sal_score': 1.6682748794555664},
 {'step_idx': 14, 'sal_score': 1.7493618726730347},
 {'step_idx': 4, 'sal_score': 2.4319353103637695}]

In [56]:
[x for x in example['logs'] if x['step_idx'] == mistake_step]

[]

In [24]:
def print_diagnostics(
    store:    GradientStore,
    scores:   torch.Tensor,
    ref_grad: torch.Tensor,
):
    """Print diagnostic info using vectorised GPU ops.
 
    Assumes store.G is already centered. Reconstructs raw norms via
    un-centering on the fly.
    """
    T = scores.shape[0]
 
    # ── Raw L1 norms (un-center to get original gradients) ───────────
    raw_l1 = (store.G + ref_grad.unsqueeze(0)).abs().sum(dim=1)   # (T,)
 
    # ── Move to numpy ────────────────────────────────────────────────
    sal_np = scores.cpu().numpy()
    raw_np = raw_l1.cpu().numpy()
 
    # ── Mistake mask (vectorised) ────────────────────────────────────
    mistake_mask = np.array([e.is_mistake for e in store.index], dtype=bool)
 
    # ── Pearson correlation ──────────────────────────────────────────
    corr = np.corrcoef(sal_np, raw_np)[0, 1]
    print(f"\n  ── Diagnostics ──")
    print(f"  Pearson(SAL, raw_L1):     {corr:.4f}")
 
    # ── Mean SAL for mistake vs normal ───────────────────────────────
    if mistake_mask.any() and (~mistake_mask).any():
        mean_mistake = sal_np[mistake_mask].mean()
        mean_normal  = sal_np[~mistake_mask].mean()
        print(f"  Mean SAL (mistake steps): {mean_mistake:.6f}")
        print(f"  Mean SAL (normal steps):  {mean_normal:.6f}")
        print(f"  Ratio (mistake/normal):   "
              f"{mean_mistake / max(mean_normal, 1e-12):.2f}")
 
    # ── Quick Acc@1 per trajectory (vectorised via slicing) ──────────
    correct = 0
    total = 0
    for traj_idx, (start, end) in enumerate(store.traj_ranges):
        if start == end:
            continue
        total += 1
        best_local = scores[start:end].argmin().item()
        predicted_step = store.index[start + best_local].step_idx
        mistake_step = int(store.traj_meta[traj_idx]["mistake_step"])
        if predicted_step == mistake_step:
            correct += 1
 
    print(f"  Quick Acc@1 (argmax):     {correct}/{total} = "
          f"{100 * correct / max(total, 1):.1f}%")

In [25]:
print_diagnostics(store, scores, ref_grad)


  ── Diagnostics ──
  Pearson(SAL, raw_L1):     0.8494
  Mean SAL (mistake steps): 3.543412
  Mean SAL (normal steps):  7.813261
  Ratio (mistake/normal):   0.45
  Quick Acc@1 (argmax):     0/58 = 0.0%


In [26]:
import pandas as pd
data = pd.read_csv("ablation/llama-3.1-8b/aggregated-results/algorithm-generated_k1_l1_norm.tsv", sep="\t")
sorted_data = data.sort_values(
    ["strategy", "step_acc", "agent_acc", "config"],
    ascending=[True, False, False, False] 
)
sorted_data.to_csv("z_tmp.tsv", sep="\t", index=False, float_format="%.4f")

In [41]:
import pandas as pd
# data = pd.read_csv("ablation/llama-3.1-8b/aggregated-results/algorithm-generated_k1_l2_norm.tsv", sep="\t")
data = pd.read_csv("ablation/llama-3.1-8b/aggregated-results/hand-crafted_k1_l2_norm.tsv", sep="\t")

# data = pd.read_csv("ablation/qwen3-8b/aggregated-results/algorithm-generated_k1_l2_norm.tsv", sep="\t")
# data = pd.read_csv("ablation/qwen3-8b/aggregated-results/hand-crafted_k1_l2_norm.tsv", sep="\t")

sorted_data = data.sort_values(
    ["step_acc", "agent_acc", "strategy", "config"],
    ascending=[False, False, False, False] 
)
sorted_data.to_csv("z_tmp.tsv", sep="\t", index=False, float_format="%.4f")

In [30]:
sorted_data = data[data['strategy'].isin(['attn_weights', 'mlp_weights'])].sort_values(
    ["step_acc", "agent_acc", "config"],
    ascending=[False, False, False]
)

sorted_data.to_csv("z_tmp.tsv", sep="\t", index=False, float_format="%.4f")

In [31]:
sorted_data = data[data['strategy'].isin(['mlp', 'attn'])].sort_values(
    ["step_acc", "agent_acc", "config"],
    ascending=[False, False, False]
)

sorted_data.to_csv("z_tmp.tsv", sep="\t", index=False, float_format="%.4f")

In [1]:
# ── Fast proxy via saved JSON outputs ────────────────────────────────────────
import json
from pathlib import Path
from core.data import _serialize_turns, select_context

OUTPUT_DIRS = {
    "hand-crafted":        Path("outputs/qwen3-8b/grad-norm/hand-crafted"),
    "algorithm-generated": Path("outputs/qwen3-8b/grad-norm/algorithm-generated"),
}
# Rough chars-per-token ratio; adjust to your tokenizer
CHARS_PER_TOKEN = 3.5
MAX_TOKENS      = 8192

for subset, out_dir in OUTPUT_DIRS.items():
    files = sorted(out_dir.glob("*.json"))
    total = truncated = 0
    for f in files:
        data = json.loads(f.read_text())
        history = data["steps"]
        for log in data["logs"]:
            step_idx    = int(log["step_idx"])
            ctx_indices = select_context(history, step_idx)
            ctx_chars   = len(_serialize_turns(history, ctx_indices))
            estimated_tokens = ctx_chars / CHARS_PER_TOKEN
            total     += 1
            truncated += int(estimated_tokens > MAX_TOKENS)
    print(f"[{subset}]  truncated (estimated): {truncated} / {total}  "
          f"({100 * truncated / max(total, 1):.1f}%)")

[hand-crafted]  truncated (estimated): 244 / 2993  (8.2%)
[algorithm-generated]  truncated (estimated): 21 / 1099  (1.9%)
